In [ ]:
import re
from docx import Document
import pandas as pd
import os

# === 1. Đọc nội dung Word ===
def extract_text_from_docx(path):
    doc = Document(path)
    paragraphs = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if text:
            paragraphs.append(text)
    return "\n".join(paragraphs), doc


# === 2. Tách phần đề và phần đáp án ===
def split_sections(text):
    marker = "ĐÁP ÁN VÀ HƯỚNG DẪN GIẢI CHI TIẾT"
    if marker in text:
        idx = text.find(marker)
        return text[:idx], text[idx:]
    return text, ""


# === 3. Tách câu hỏi và các lựa chọn (A–D) ===
def extract_questions(de_text):
    """
    Trích danh sách câu hỏi từ phần đề, chỉ lấy nội dung câu hỏi (không lấy lựa chọn A–D)
    """
    # regex tìm từng khối câu hỏi: "Câu 1: ...", "Câu 2: ..."
    pattern = r"Câu\s+(\d+)[\.:]?\s*([\s\S]*?)(?=\n\s*Câu\s+\d+[\.:]?|\Z|PHẦN|$)"
    matches = re.findall(pattern, de_text)
    
    questions = []
    for num, qtext in matches:
        # làm sạch xuống dòng và khoảng trắng  
        questions.append({
            "id": int(num),
            "question_text": qtext
        })
    return questions


def extract_answers_from_docx(doc):
    import re

    answers = []
    pattern = r"Câu\s+(\d+)"
    current_id = None
    current_text = []
    current_underline = None
    inside_answer_section = False

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue

        # Bắt đầu đọc khi gặp đoạn "ĐÁP ÁN VÀ HƯỚNG DẪN GIẢI"
        if not inside_answer_section:
            if "ĐÁP ÁN" in text.upper() or "HƯỚNG DẪN" in text.upper():
                inside_answer_section = True
            continue

        # Kiểm tra có phải đoạn numbering (Câu mới)
        is_numbered = para._p.pPr is not None and para._p.pPr.numPr is not None
        m = re.match(pattern, text)

        # Nếu là đoạn bắt đầu câu hỏi mới
        if m or is_numbered:
            if current_id is not None:
                # Lấy hướng dẫn giải 
                full_text = "\n".join(current_text).strip()
                match = re.search(r"hướng\s*dẫn\s*giải[:\-]?", full_text, re.IGNORECASE)
                if match:
                    full_text = full_text[match.end():].strip()  # lấy sau cụm đó

                final = "Đáp án: " + (current_underline or "") + "\n" + (full_text or "")

                answers.append({
                    "id": current_id,
                    "answer": final
                })

            if m:
                current_id = int(m.group(1))
            else:
                current_id = (current_id or 0) + 1

            current_text = []
            current_underline = None
            continue

        # Kiểm tra đáp án
        underline_letters = []
        for run in para.runs:
            if run.font.underline and re.match(r"^[A-D]\.?$", run.text.strip()):
                underline_letters.append(run.text.strip().replace('.', ''))
        if underline_letters:
            current_underline = underline_letters[0]

        current_text.append(text)

    # Lưu câu cuối cùng
    if current_id is not None:
        full_text = "\n".join(current_text).strip()
        match = re.search(r"hướng\s*dẫn\s*giải[:\-]?", full_text, re.IGNORECASE)
        if match:
            full_text = full_text[match.end():].strip()

        final = "Đáp án: " + (current_underline or "") + "\n" + (full_text or "")

        answers.append({
            "id": current_id,
            "answer": final
        })

    return answers


# === 5. Gộp câu hỏi và đáp án ===
def merge_q_and_a(questions, answers):
    merged = []
    for q in questions:
        ans = next((a for a in answers if a["id"] == q["id"]), None)
        merged.append({
            "id": q["id"],
            "question_text": q["question_text"],
            "answer": ans["answer"] if ans else None,
        })
    return merged


folder = "test_data"
all_data = []

# Duyệt qua tất cả file trong thư mục
for filename in os.listdir(folder):
    if filename.endswith(".docx"):
        path = os.path.join(folder, filename)

        try:
            text, doc = extract_text_from_docx(path)
            de, dap_an_text = split_sections(text)
            questions = extract_questions(de)
            answers = extract_answers_from_docx(doc)
            merged_data = merge_q_and_a(questions, answers)
            all_data.extend(merged_data)
        except Exception as e:
            print(f"⚠️ Lỗi khi xử lý {filename}: {e}")

df = pd.DataFrame(all_data)
df.to_excel("data.xlsx", index=False)

print("✅ Đã xử lý xong tất cả file. Kết quả lưu tại data.xlsx")



📄 Đang xử lý: 01_TdtntTayNguyen_Ntt.docx
📄 Đang xử lý: 03_Thpt-Victory_ThptNguyenVanCu-LongChau.docx
📄 Đang xử lý: 07_ThptCbq_ThptTdn-PhongBaoBui.docx
📄 Đang xử lý: 08ThptTranHungDao-ThptChuVanAn-PhamThanh.docx
📄 Đang xử lý: 10_TruongThptDtntN_TrangLong_ThptVietDuc-HuychungNguyen.docx
📄 Đang xử lý: 12_Thpt_EaSup(DeNopL2)-KhongtenKhong.docx
✅ Đã xử lý xong tất cả file. Kết quả lưu tại merged_questions_answers.xlsx


In [3]:
from docx import Document

def check_paragraph_numbering(path):
    doc = Document(path)
    results = []

    for i, para in enumerate(doc.paragraphs):
        text = para.text.strip()
        if not text:
            continue

        # Kiểm tra cách 1: có từ "Câu" trong văn bản
        has_word_cau = bool(re.match(r"^Câu\s+\d+", text))

        # Kiểm tra cách 2: có sử dụng numbering format
        # (paragraph có property style dạng danh sách)
        is_numbered = para._p.pPr is not None and para._p.pPr.numPr is not None

        results.append({
            "index": i,
            "text": text[:80],  # cắt ngắn cho dễ xem
            "has_word_cau": has_word_cau,
            "is_numbered": is_numbered,
            "style": para.style.name if para.style else None
        })

    return results


# ==== TEST ====
if __name__ == "__main__":
    path = "test_data/03_Thpt-Victory_ThptNguyenVanCu-LongChau.docx"  # ← thay bằng đường dẫn thực tế
    data = check_paragraph_numbering(path)

    for item in data:
        print(f"{item['index']:03d} | has_word_cau={item['has_word_cau']} | "
              f"is_numbered={item['is_numbered']} | style={item['style']} | "
              f"text={item['text']}")


000 | has_word_cau=False | is_numbered=False | style=Normal | text=Họ, tên thí sinh:
001 | has_word_cau=False | is_numbered=False | style=Normal | text=Số báo danh:……………………………………………………………….
003 | has_word_cau=False | is_numbered=False | style=Normal | text=Phần I: Câu trắc nghiệm nhiều lựa chọn
004 | has_word_cau=True | is_numbered=False | style=Normal | text=Câu 1. Chất lỏng và chất khí có chung đặc điểm nào sau đây?
005 | has_word_cau=False | is_numbered=False | style=Normal | text=A. Luôn chiếm đầy thể tích bình chứa.	B. Không nén được.
006 | has_word_cau=False | is_numbered=False | style=Normal | text=C. Có thể chảy thành dòng.		D. Có các phân tử ở những vị trí cố định.
007 | has_word_cau=True | is_numbered=False | style=Normal | text=Câu 2. Một quả bong bóng chứa một lượng không khí nhất định. Cách nào sau đây là
008 | has_word_cau=False | is_numbered=False | style=Normal | text=A. Đặt quả bong bóng vào ngăn mát tủ lạnh.
009 | has_word_cau=False | is_numbered=False | style=Normal 